In [1]:
%pip install -q duckdb

In [2]:
import duckdb

con = duckdb.connect("online_retail.duckdb")

print("DuckDB version:", duckdb.__version__)
print("Connection successful!")

DuckDB version: 1.3.2
Connection successful!


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
from pathlib import Path

data_dir = Path("/content/drive/MyDrive/MVJ_Internship/Data")

print("Folder exists:", data_dir.exists())
print("\nCSV files found:")

for file in sorted(data_dir.glob("*.csv")):
    print(file.name, "|", round(file.stat().st_size / (1024 * 1024), 2), "MB")

Folder exists: True

CSV files found:
online_retail_II_csv(2009-2010).csv | 38.82 MB
online_retail_II_csv(2010-2011).csv | 39.38 MB


In [5]:
%pip install -q duckdb

In [6]:
import duckdb
from pathlib import Path

con = duckdb.connect("online_retail.duckdb")

csv_files = sorted(data_dir.glob("*.csv"))

if len(csv_files) < 2:
    raise FileNotFoundError(
        f"Expected at least 2 CSV files in {data_dir}, "
        f"but found {len(csv_files)}."
    )

for file in csv_files:
    print(f"\nFile: {file.name}")

    # Read the CSV directly without modifying the source file
    result = con.execute(
        "SELECT COUNT(*) FROM read_csv_auto(?, header=true)",
        [str(file)]
    ).fetchone()

    print("Data rows:", f"{result[0]:,}")


File: online_retail_II_csv(2009-2010).csv
Data rows: 525,461

File: online_retail_II_csv(2010-2011).csv
Data rows: 541,910


In [7]:
for file in csv_files:
    print(f"\n{'=' * 50}")
    print("FILE:", file.name)

    print("\nColumn names and data types:")
    print(
        con.execute(
            "DESCRIBE SELECT * FROM read_csv_auto(?, header=true)",
            [str(file)]
        ).fetchdf().to_string(index=False)
    )

    print("\nFirst 5 rows:")
    print(
        con.execute(
            "SELECT * FROM read_csv_auto(?, header=true) LIMIT 5",
            [str(file)]
        ).fetchdf().to_string(index=False)
    )


FILE: online_retail_II_csv(2009-2010).csv

Column names and data types:
column_name column_type null  key default extra
    Invoice     VARCHAR  YES None    None  None
  StockCode     VARCHAR  YES None    None  None
Description     VARCHAR  YES None    None  None
   Quantity      BIGINT  YES None    None  None
InvoiceDate        DATE  YES None    None  None
      Price      DOUBLE  YES None    None  None
Customer ID      BIGINT  YES None    None  None
    Country     VARCHAR  YES None    None  None

First 5 rows:
Invoice StockCode                         Description  Quantity InvoiceDate  Price  Customer ID        Country
 489434     85048 15CM CHRISTMAS GLASS BALL 20 LIGHTS        12  2001-12-09   6.95        13085 United Kingdom
 489434    79323P                  PINK CHERRY LIGHTS        12  2001-12-09   6.75        13085 United Kingdom
 489434    79323W                 WHITE CHERRY LIGHTS        12  2001-12-09   6.75        13085 United Kingdom
 489434     22041        RECORD FRAM

In [8]:
#combine both csv files into one duckDB view
from pathlib import Path

assert len(csv_files) == 2, (
    f"Expected exactly 2 CSV files, found {len(csv_files)}: "
    f"{[f.name for f in csv_files]}"
)

# Confirm that both files have the same column names
column_lists = []

for file in csv_files:
    columns = con.execute(
        "DESCRIBE SELECT * FROM read_csv_auto(?, header=true)",
        [str(file)]
    ).fetchdf()["column_name"].tolist()

    column_lists.append(columns)

    print(file.name, "->", columns)

if column_lists[0] != column_lists[1]:
    raise ValueError(
        "The two CSV files have different column names or column order. "
        "Stop here and share the output above."
    )

# Create a view over both files
file_a = csv_files[0].as_posix()
file_b = csv_files[1].as_posix()

# Using f-strings because DDL queries (CREATE VIEW) cannot accept prepared parameters
con.execute(f"""
    CREATE OR REPLACE VIEW online_retail AS
    SELECT * FROM read_csv_auto('{file_a}', header=true)
    UNION ALL
    SELECT * FROM read_csv_auto('{file_b}', header=true)
""")

print("Combined view created successfully.")

online_retail_II_csv(2009-2010).csv -> ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']
online_retail_II_csv(2010-2011).csv -> ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']
Combined view created successfully.


In [9]:
#verify the combined dataset
print("Combined data rows:")
print(f"{con.execute('SELECT COUNT(*) FROM online_retail').fetchone()[0]:,}")

print("\nColumn names and types:")
print(
    con.execute("DESCRIBE online_retail")
    .fetchdf()
    .to_string(index=False)
)

print("\nSample records:")
print(
    con.execute("SELECT * FROM online_retail LIMIT 5")
    .fetchdf()
    .to_string(index=False)
)

Combined data rows:
1,067,371

Column names and types:
column_name column_type null  key default extra
    Invoice     VARCHAR  YES None    None  None
  StockCode     VARCHAR  YES None    None  None
Description     VARCHAR  YES None    None  None
   Quantity      BIGINT  YES None    None  None
InvoiceDate        DATE  YES None    None  None
      Price      DOUBLE  YES None    None  None
Customer ID      BIGINT  YES None    None  None
    Country     VARCHAR  YES None    None  None

Sample records:
Invoice StockCode                         Description  Quantity InvoiceDate  Price  Customer ID        Country
 489434     85048 15CM CHRISTMAS GLASS BALL 20 LIGHTS        12  2001-12-09   6.95        13085 United Kingdom
 489434    79323P                  PINK CHERRY LIGHTS        12  2001-12-09   6.75        13085 United Kingdom
 489434    79323W                 WHITE CHERRY LIGHTS        12  2001-12-09   6.75        13085 United Kingdom
 489434     22041        RECORD FRAME 7" SINGLE SIZE

In [10]:
#run basuc data quality checks
quality_sql = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE InvoiceDate IS NULL
    ) AS missing_dates,
    COUNT(*) FILTER (
        WHERE Quantity IS NULL
    ) AS missing_quantities,
    COUNT(*) FILTER (
        WHERE Price IS NULL
    ) AS missing_prices,
    COUNT(*) FILTER (
        WHERE "Customer ID" IS NULL
    ) AS missing_customer_ids,
    COUNT(DISTINCT Invoice) AS distinct_invoices,
    MIN(InvoiceDate) AS first_transaction,
    MAX(InvoiceDate) AS last_transaction
FROM online_retail
"""

quality = con.execute(quality_sql).fetchdf()
display(quality)

print("Cancellation rows:")
print(con.execute("""
    SELECT COUNT(*)
    FROM online_retail
    WHERE starts_with(upper(trim(CAST(Invoice AS VARCHAR))), 'C')
""").fetchone()[0])

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,missing_dates,missing_quantities,missing_prices,missing_customer_ids,distinct_invoices,first_transaction,last_transaction
0,1067371,0,0,0,243007,53628,2001-02-10,2031-10-10


Cancellation rows:
19494


In [11]:
#create a consistent analysis dataset
con.execute("""
    CREATE OR REPLACE VIEW retail_sales AS
    SELECT
        *,
        CAST(Quantity AS DOUBLE) * CAST(Price AS DOUBLE) AS Revenue
    FROM online_retail
    WHERE InvoiceDate IS NOT NULL
      AND Quantity > 0
      AND Price > 0
      AND NOT starts_with(
          upper(trim(CAST(Invoice AS VARCHAR))), 'C'
      )
""")

print("Original rows:", f"{con.execute('SELECT COUNT(*) FROM online_retail').fetchone()[0]:,}")
print("Rows retained for sales analysis:", f"{con.execute('SELECT COUNT(*) FROM retail_sales').fetchone()[0]:,}")
print("Excluded rows:", f"{con.execute('SELECT COUNT(*) FROM online_retail').fetchone()[0] - con.execute('SELECT COUNT(*) FROM retail_sales').fetchone()[0]:,}")


Original rows: 1,067,371
Rows retained for sales analysis: 1,041,670
Excluded rows: 25,701


Day 5 **Deliverables**

In [12]:
monthly_sql = """
SELECT
    strftime(InvoiceDate, '%Y-%m') AS sales_month,
    COUNT(DISTINCT Invoice) AS total_orders,
    ROUND(SUM(Revenue), 2) AS total_revenue,
    ROUND(
        SUM(Revenue) / NULLIF(COUNT(DISTINCT Invoice), 0),
        2
    ) AS average_order_value
FROM retail_sales
GROUP BY sales_month
ORDER BY sales_month
"""

monthly_kpis = con.execute(monthly_sql).fetchdf()
display(monthly_kpis)

,sales_month,total_orders,total_revenue,average_order_value
0,2001-02,66,19831.43,300.48
1,2001-03,58,25831.21,445.37
2,2001-04,79,30823.15,390.17
3,2001-06,47,15732.15,334.73
4,2001-07,85,30314.81,356.64
...,...,...,...,...
295,2030-11,123,60029.08,488.04
296,2031-01,53,18704.68,352.92
297,2031-03,75,29718.33,396.24
298,2031-08,63,35964.47,570.86


In [13]:
#country wise performance
country_sql = """
SELECT
    Country,
    COUNT(DISTINCT Invoice) AS total_orders,
    ROUND(SUM(Revenue), 2) AS total_revenue,
    ROUND(
        SUM(Revenue) / NULLIF(COUNT(DISTINCT Invoice), 0),
        2
    ) AS average_order_value
FROM retail_sales
GROUP BY Country
ORDER BY total_revenue DESC
"""

country_kpis = con.execute(country_sql).fetchdf()
display(country_kpis)

,Country,total_orders,total_revenue,average_order_value
0,United Kingdom,36535,17870977.78,489.15
1,EIRE,626,664431.78,1061.39
2,Netherlands,228,554232.34,2430.84
3,Germany,789,431262.46,546.59
4,France,622,356944.60,573.87
5,Australia,95,169968.11,1789.14
6,Spain,154,109178.53,708.95
7,Switzerland,93,101011.29,1086.14
8,Sweden,105,91903.72,875.27
9,Denmark,43,69862.19,1624.70


In [14]:
#monthly performance by country
monthly_country_sql = """
SELECT
    strftime(InvoiceDate, '%Y-%m') AS sales_month,
    Country,
    COUNT(DISTINCT Invoice) AS total_orders,
    ROUND(SUM(Revenue), 2) AS total_revenue,
    ROUND(
        SUM(Revenue) / NULLIF(COUNT(DISTINCT Invoice), 0),
        2
    ) AS average_order_value
FROM retail_sales
GROUP BY sales_month, Country
ORDER BY sales_month, total_revenue DESC
"""

monthly_country_kpis = con.execute(monthly_country_sql).fetchdf()
display(monthly_country_kpis)

,sales_month,Country,total_orders,total_revenue,average_order_value
0,2001-02,United Kingdom,57,16315.82,286.24
1,2001-02,Malta,2,1636.38,818.19
2,2001-02,Germany,2,824.20,412.10
3,2001-02,Spain,2,492.04,246.02
4,2001-02,Portugal,2,403.33,201.67
...,...,...,...,...,...
1623,2031-08,France,2,251.02,125.51
1624,2031-08,Netherlands,1,147.09,147.09
1625,2031-10,United Kingdom,80,29245.82,365.57
1626,2031-10,Iceland,1,611.53,611.53


In [15]:
#export the results to drive
from pathlib import Path

output_dir = Path(
    "/content/drive/MyDrive/MVJ_Internship/Project1/sql"
)
output_dir.mkdir(parents=True, exist_ok=True)

monthly_kpis.to_csv(output_dir / "monthly_kpis.csv", index=False)
country_kpis.to_csv(output_dir / "country_kpis.csv", index=False)
monthly_country_kpis.to_csv(
    output_dir / "monthly_country_kpis.csv",
    index=False
)

print("Exported files:")
for file in sorted(output_dir.glob("*.csv")):
    print(file.name, "-", file.stat().st_size, "bytes")

Exported files:
country_kpis.csv - 1254 bytes
monthly_country_kpis.csv - 55496 bytes
monthly_kpis.csv - 8175 bytes


In [16]:
#save the sql queries
queries_sql = """-- MVJ Internship: Day 5 SQL Aggregation Analysis
-- Dataset: UCI Online Retail II
-- Initial sales filter excludes cancellations, non-positive quantity/price,
-- and missing invoice dates.

-- 1. Data quality
""" + quality_sql + """;

-- 2. Monthly KPIs
""" + monthly_sql + """;

-- 3. Country KPIs
""" + country_sql + """;

-- 4. Monthly country KPIs
""" + monthly_country_sql + """;
"""

(output_dir / "queries.sql").write_text(
    queries_sql,
    encoding="utf-8"
)

print("Saved:", output_dir / "queries.sql")

Saved: /content/drive/MyDrive/MVJ_Internship/Project1/sql/queries.sql


In [17]:
#identify 3 real findings
print("TOP 5 COUNTRIES BY REVENUE")
display(country_kpis.head(5))

print("TOP 5 MONTHS BY REVENUE")
display(
    monthly_kpis.sort_values(
        "total_revenue", ascending=False
    ).head(5)
)

print("MONTH WITH THE MOST ORDERS")
display(
    monthly_kpis.sort_values(
        "total_orders", ascending=False
    ).head(1)
)

TOP 5 COUNTRIES BY REVENUE


,Country,total_orders,total_revenue,average_order_value
0,United Kingdom,36535,17870977.78,489.15
1,EIRE,626,664431.78,1061.39
2,Netherlands,228,554232.34,2430.84
3,Germany,789,431262.46,546.59
4,France,622,356944.60,573.87


TOP 5 MONTHS BY REVENUE


,sales_month,total_orders,total_revenue,average_order_value
108,2011-11,2898,1573737.31,543.04
107,2011-10,2131,1198489.17,562.41
106,2011-09,1837,1058590.17,576.26
97,2010-12,1651,868188.25,525.86
103,2011-06,1619,799542.15,493.85


MONTH WITH THE MOST ORDERS


,sales_month,total_orders,total_revenue,average_order_value
108,2011-11,2898,1573737.31,543.04


In [18]:
print("DuckDB connection:", con)
print(con.execute("SHOW TABLES").fetchdf())

DuckDB connection: <duckdb.duckdb.DuckDBPyConnection object at 0x7c230820a370>
            name
0  online_retail
1   retail_sales


In [19]:
#check cust_id and date fields
con.execute("""
    SELECT
        COUNT(*) AS sales_rows,
        COUNT(*) FILTER (
            WHERE TRY_CAST("Customer ID" AS BIGINT) IS NULL
        ) AS rows_without_valid_customer_id,
        COUNT(DISTINCT TRY_CAST("Customer ID" AS BIGINT))
            AS identified_customers,
        MIN(InvoiceDate) AS first_date,
        MAX(InvoiceDate) AS last_date
    FROM retail_sales
""").df()

,sales_rows,rows_without_valid_customer_id,identified_customers,first_date,last_date
0,1041670,236121,5878,2001-02-10,2031-10-10


In [20]:
#pracrice inner join
con.execute("""
    CREATE OR REPLACE TEMP VIEW customer_spending AS
    SELECT
        TRY_CAST("Customer ID" AS BIGINT) AS customer_id,
        ROUND(SUM(Revenue), 2) AS total_revenue
    FROM retail_sales
    WHERE TRY_CAST("Customer ID" AS BIGINT) IS NOT NULL
    GROUP BY TRY_CAST("Customer ID" AS BIGINT);

    CREATE OR REPLACE TEMP VIEW customer_orders AS
    SELECT
        TRY_CAST("Customer ID" AS BIGINT) AS customer_id,
        COUNT(DISTINCT Invoice) AS total_orders
    FROM retail_sales
    WHERE TRY_CAST("Customer ID" AS BIGINT) IS NOT NULL
    GROUP BY TRY_CAST("Customer ID" AS BIGINT);
""")

join_practice_sql = """
SELECT
    s.customer_id,
    s.total_revenue,
    o.total_orders
FROM customer_spending AS s
INNER JOIN customer_orders AS o
    ON s.customer_id = o.customer_id
ORDER BY s.total_revenue DESC
LIMIT 10
"""

display(con.execute(join_practice_sql).df())


,customer_id,total_revenue,total_orders
0,18102,608821.65,145
1,14646,528602.52,151
2,14156,313946.37,156
3,14911,295972.63,398
4,17450,246973.09,51
5,13694,196482.81,143
6,17511,175603.55,60
7,16446,168472.50,2
8,16684,147142.77,55
9,12415,144458.37,28


In [21]:
#build the complete customer-level table
#Now calculate spending, orders, units, and purchase history, then join the summaries into one row per customer
customer_table_sql = """
WITH customer_metrics AS (
    SELECT
        TRY_CAST("Customer ID" AS BIGINT) AS customer_id,
        ROUND(SUM(Revenue), 2) AS total_revenue,
        COUNT(DISTINCT Invoice) AS total_orders,
        SUM(Quantity) AS total_units,
        ROUND(AVG(Revenue), 2) AS average_line_revenue
    FROM retail_sales
    WHERE TRY_CAST("Customer ID" AS BIGINT) IS NOT NULL
    GROUP BY TRY_CAST("Customer ID" AS BIGINT)
),
purchase_history AS (
    SELECT
        TRY_CAST("Customer ID" AS BIGINT) AS customer_id,
        MIN(InvoiceDate) AS first_purchase_date,
        MAX(InvoiceDate) AS latest_purchase_date
    FROM retail_sales
    WHERE TRY_CAST("Customer ID" AS BIGINT) IS NOT NULL
    GROUP BY TRY_CAST("Customer ID" AS BIGINT)
)
SELECT
    m.customer_id,
    m.total_revenue,
    m.total_orders,
    m.total_units,
    m.average_line_revenue,
    p.first_purchase_date,
    p.latest_purchase_date
FROM customer_metrics AS m
INNER JOIN purchase_history AS p
    ON m.customer_id = p.customer_id
"""

con.execute("""
    CREATE OR REPLACE TEMP VIEW customer_level AS
""" + customer_table_sql)

display(
    con.execute("""
        SELECT *
        FROM customer_level
        ORDER BY total_revenue DESC
        LIMIT 10
    """).df()
)

,customer_id,total_revenue,total_orders,total_units,average_line_revenue,first_purchase_date,latest_purchase_date
0,18102,608821.65,145,188340.0,575.45,2001-08-10,2031-08-10
1,14646,528602.52,151,367193.0,137.34,2001-04-10,2030-11-10
2,14156,313946.37,156,165992.0,77.56,2001-12-09,2030-09-10
3,14911,295972.63,398,149987.0,26.32,2001-07-10,2031-08-10
4,17450,246973.09,51,84720.0,582.48,2007-12-10,2028-10-10
5,13694,196482.81,143,189205.0,128.84,2001-03-10,2030-09-10
6,17511,175603.55,60,119656.0,91.89,2001-09-10,2029-09-10
7,16446,168472.50,2,80997.0,56157.50,2011-05-18,2011-12-09
8,16684,147142.77,55,104810.0,204.93,2001-02-10,2031-10-10
9,12415,144458.37,28,91447.0,156.00,2004-11-10,2030-07-10


In [22]:
#Add customer segments using CASE

In [26]:
# #We'll classify customers into illustrative spending bands:
# - High Value: total revenue of 5,000 or more
# - Medium Value: total revenue from 1,000 up to 4,999.99
# - Low Value: total revenue below 1,000
segmentation_sql = """
SELECT
    *,
    CASE
        WHEN total_revenue >= 5000 THEN 'High Value'
        WHEN total_revenue >= 1000 THEN 'Medium Value'
        ELSE 'Low Value'
    END AS customer_segment
FROM customer_level
"""

con.execute("""
    CREATE OR REPLACE TEMP VIEW customer_segments AS
""" + segmentation_sql)

display(con.execute("""
    SELECT *
    FROM customer_segments
    ORDER BY total_revenue DESC
    LIMIT 15
""").df())

,customer_id,total_revenue,total_orders,total_units,average_line_revenue,first_purchase_date,latest_purchase_date,customer_segment
0,18102,608821.65,145,188340.0,575.45,2001-08-10,2031-08-10,High Value
1,14646,528602.52,151,367193.0,137.34,2001-04-10,2030-11-10,High Value
2,14156,313946.37,156,165992.0,77.56,2001-12-09,2030-09-10,High Value
3,14911,295972.63,398,149987.0,26.32,2001-07-10,2031-08-10,High Value
4,17450,246973.09,51,84720.0,582.48,2007-12-10,2028-10-10,High Value
5,13694,196482.81,143,189205.0,128.84,2001-03-10,2030-09-10,High Value
6,17511,175603.55,60,119656.0,91.89,2001-09-10,2029-09-10,High Value
7,16446,168472.50,2,80997.0,56157.50,2011-05-18,2011-12-09,High Value
8,16684,147142.77,55,104810.0,204.93,2001-02-10,2031-10-10,High Value
9,12415,144458.37,28,91447.0,156.00,2004-11-10,2030-07-10,High Value


In [25]:
#Summarize the customer segments
segment_summary_sql = """
SELECT
    customer_segment,
    COUNT(*) AS customer_count,
    ROUND(SUM(total_revenue), 2) AS segment_revenue,
    ROUND(AVG(total_revenue), 2) AS average_revenue_per_customer,
    ROUND(AVG(total_orders), 2) AS average_orders_per_customer
FROM customer_segments
GROUP BY customer_segment
ORDER BY
    CASE customer_segment
        WHEN 'High Value' THEN 1
        WHEN 'Medium Value' THEN 2
        ELSE 3
    END
"""

segment_summary = con.execute(segment_summary_sql).df()
display(segment_summary)

,customer_segment,customer_count,segment_revenue,average_revenue_per_customer,average_orders_per_customer
0,High Value,670,11774223.43,17573.47,25.27
1,Medium Value,2077,4647860.72,2237.78,6.66
2,Low Value,3131,1321345.01,422.02,1.98


In [27]:
#validate the customer table
validation = con.execute("""
    SELECT
        COUNT(*) AS customer_rows,
        COUNT(DISTINCT customer_id) AS unique_customer_ids,
        COUNT(*) FILTER (
            WHERE customer_id IS NULL
        ) AS null_customer_ids,
        COUNT(*) FILTER (
            WHERE total_revenue IS NULL OR total_revenue < 0
        ) AS invalid_revenue_rows,
        COUNT(*) FILTER (
            WHERE total_orders < 1
        ) AS customers_without_orders
    FROM customer_segments
""").df()

display(validation)

,customer_rows,unique_customer_ids,null_customer_ids,invalid_revenue_rows,customers_without_orders
0,5878,5878,0,0,0


In [28]:
#export day 6 results
from pathlib import Path

day6_dir = Path(
    "/content/drive/MyDrive/MVJ_Internship/Project1/sql"
)
day6_dir.mkdir(parents=True, exist_ok=True)

con.execute("""
    SELECT * FROM customer_level
    ORDER BY customer_id
""").fetchdf().to_csv(
    day6_dir / "customer_level.csv", index=False
)

con.execute("""
    SELECT * FROM customer_segments
    ORDER BY customer_id
""").fetchdf().to_csv(
    day6_dir / "customer_segments.csv", index=False
)

segment_summary.to_csv(
    day6_dir / "customer_segment_summary.csv", index=False
)

print("Saved files:")
for name in [
    "customer_level.csv",
    "customer_segments.csv",
    "customer_segment_summary.csv"
]:
    path = day6_dir / name
    print(f"{name}: {path.stat().st_size:,} bytes")

Saved files:
customer_level.csv: 290,355 bytes
customer_segments.csv: 356,053 bytes
customer_segment_summary.csv: 227 bytes


In [29]:
#save the day 6 files
day6_sql = """-- MVJ Internship - Day 6
-- Online Retail II: SQL joins and customer segmentation
-- Prerequisite: online_retail and retail_sales views from Day 5.

-- 1. Join customer spending to order counts
""" + join_practice_sql + """;

-- 2. Customer-level table using a join
""" + customer_table_sql + """;

-- 3. CASE-based customer segmentation
""" + segmentation_sql + """;

-- 4. Customer segment summary
""" + segment_summary_sql + """;
"""

(day6_dir / "day6_customer_segmentation.sql").write_text(
    day6_sql,
    encoding="utf-8"
)

print("SQL file saved:", day6_dir / "day6_customer_segmentation.sql")

SQL file saved: /content/drive/MyDrive/MVJ_Internship/Project1/sql/day6_customer_segmentation.sql


In [30]:
#record the 3 findings
print("CUSTOMER SEGMENT SUMMARY")
display(segment_summary)

print("TOP 10 CUSTOMERS BY REVENUE")
display(con.execute("""
    SELECT customer_id, total_revenue, total_orders, customer_segment
    FROM customer_segments
    ORDER BY total_revenue DESC
    LIMIT 10
""").df())

print("CUSTOMERS WITH MULTIPLE ORDERS")
display(con.execute("""
    SELECT COUNT(*) AS customers_with_multiple_orders
    FROM customer_segments
    WHERE total_orders > 1
""").df())

CUSTOMER SEGMENT SUMMARY


,customer_segment,customer_count,segment_revenue,average_revenue_per_customer,average_orders_per_customer
0,High Value,670,11774223.43,17573.47,25.27
1,Medium Value,2077,4647860.72,2237.78,6.66
2,Low Value,3131,1321345.01,422.02,1.98


TOP 10 CUSTOMERS BY REVENUE


,customer_id,total_revenue,total_orders,customer_segment
0,18102,608821.65,145,High Value
1,14646,528602.52,151,High Value
2,14156,313946.37,156,High Value
3,14911,295972.63,398,High Value
4,17450,246973.09,51,High Value
5,13694,196482.81,143,High Value
6,17511,175603.55,60,High Value
7,16446,168472.50,2,High Value
8,16684,147142.77,55,High Value
9,12415,144458.37,28,High Value


CUSTOMERS WITH MULTIPLE ORDERS


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,customers_with_multiple_orders
0,4255
